# Big-M Elimination (Tight M / Indicator) — Before, Principle, Application, and Effect

When modeling fixed costs (e.g., facility opening, setup changeovers) with logical constraints like "can produce if opened," how you choose the M in `x <= M*y` (Big-M) drastically changes the strength of the LP relaxation. If M is loose, `x` can be almost fully utilized even if `y` takes only a slightly positive fractional value, resulting in a lower bound that hardly tightens.

This notebook tracks this phenomenon and its remedy through the following pattern:

1. **Issue (before)** — Analyze a loose Big-M with `mk.analyze` and check the weakness of the pure LP relaxation bound.
2. **Principle (principle)** — See visually how the LP relaxation region changes depending on the size of M.
3. **Application (how)** — Apply a tight M and `addConsIndicator` (SCIP Indicator constraint).
4. **Effect (before/after)** — Compare the root dual bound, gap, and node count using `mk.compare_variants`.

The subject is **Production Planning with Fixed Costs** (`samples/others/fixed_charge.py`). There are 8 facilities, and each facility can only produce `x_i>0` if it is opened (`y_i=1`). `build_model(bigm=...)` provides three variants: `"loose"` (a huge constant), `"tight"` (the maximum actual possible production), and `"indicator"` (SCIP Indicator constraint).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import fixed_charge as fc

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Issue (before) — Diagnosing a loose Big-M

First, we run `mk.analyze` on the loose Big-M version (`bigm="loose"`).

In [ ]:
report = mk.analyze(lambda: fc.build_model("loose"), name="fixed_charge-loose", time_limit=10)
print(report.summary())

For this scale (8 facilities), presolve automatically tightens loose Big-M, so `numerical_scale` **does not** fire (the residual coefficient scale is designed to be checked after presolve — see "When It Is Ineffective" in [3. Big-M Elimination](../../playbook/03-bigm.md)). We will directly confirm the weakness that cannot be seen by diagnosis alone by checking the **pure LP relaxation bound** with presolve/cuts disabled.

In [ ]:
for bigm in ("loose", "tight", "indicator"):
    print(f"{bigm:10s}: 純粋LP緩和境界 = {fc.lp_relaxation_bound(bigm):.1f}")

Loose Big-M has a significantly lower LP relaxation bound (weaker lower bound). In a normal solve with presolve/cuts enabled, much of this difference is automatically bridged, but there is a clear difference in the "quality of the formulation itself."
Next, we visually see the principle behind this.

## 2. Principle (principle) — A Loose Big-M Widens the LP Relaxation Region

Consider the linking constraint `x <= M*y` for a single facility (where `y` is the continuous relaxation of the binary `0<=y<=1`, and `x` is `0<=x<=cap`).
When M is the truly required value (`cap`, tight), making `y` slightly positive only allows `x` to be used by that amount (the region is below the line from the origin to `(cap, 1)`). When M is huge (loose), even if `y` is just slightly positive, `x` can be used up to its upper bound `cap` — the LP relaxation permits an unrealistic solution of "full production with a negligible opening cost."

In [ ]:
cap = 70.0  # F1の容量
tight_m = 70.0  # min(容量, 総需要) = tight M
mids = [500.0, 5000.0]
loose_m = 100000.0

def hex_to_rgba(h, alpha):
    h = h.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

y = np.linspace(0, 1, 400)
Ms = [(tight_m, "tight M = 70 (真に必要な最小値)", C["s1"]),
      (500.0, "M = 500", C["s2"]),
      (5000.0, "M = 5000", C["warn"]),
      (loose_m, "loose M = 100000", C["muted"])]

fig = go.Figure(layout=base_layout(
    "連動制約 x <= M・y の LP緩和領域(y:施設開設の連続緩和, x:生産量)",
    "y(開設、連続緩和)", "x(生産量、上限cap=70)", height=440))
for m_val, label, col in Ms:
    upper = np.minimum(cap, m_val * y)
    fig.add_trace(go.Scatter(x=np.r_[y, y[::-1]], y=np.r_[upper, np.zeros_like(y)],
        fill="toself", fillcolor=hex_to_rgba(col, 0.10),
        line=dict(color=col, width=2.5), name=label,
        hovertemplate=label + ": x<=%{y:.1f}<extra></extra>"))
fig.update_yaxes(range=[0, cap * 1.05])
# y=0.01 という「ほぼ閉鎖」でもloose Mならxがフル生産できてしまう点を強調
fig.add_annotation(x=0.02, y=cap * 0.98, ax=0.25, ay=cap * 0.98,
    text="y=0.02 でも loose M なら x はほぼ cap まで使える(緩和の「ただ乗り」)",
    showarrow=True, arrowhead=2, arrowcolor=C["warn"],
    font=dict(color=C["warn"], size=11))
show(fig)

Tight M (blue) is exactly the line from the origin to `(1, cap)`, yielding a "lean" relaxation where production is strictly proportional to the value of `y`. As M increases (green → orange → gray), the curve sticks to the upper left, widening the region where `x` can be fully utilized even if `y` is almost 0 — this is the true cause of lowering the LP relaxation bound.

**Indicator constraints eliminate this M selection problem entirely.** The logic `y=0 ⟹ x<=0` is passed directly to the solver, and during branching, it is tightened by specialized separation routines (indicator cuts) that handle logical constraints. The user no longer needs to choose a value for M.

## 3. Application (how) — Tight M and `addConsIndicator`

There is no general-purpose helper; we use PySCIPOpt directly.

```python
# tight M: Restrict M to the minimum value derivable from variable bounds
m.addCons(x[i] <= min(cap[i], demand) * y[i], name=f"link_{i}")

# Indicator constraint: x<=0 when y=0 (activeone=False sets the orientation to "constraint is active when binvar=0")
m.addConsIndicator(x <= 0, binvar=y[i], activeone=False, name=f"link_{i}")
```

Below is a minimal working check. We verify that all three formulations reach the same optimal value.

In [ ]:
from pyscipopt import Model, quicksum

def toy_model(bigm: str) -> Model:
    m = Model(); m.hideOutput()
    cap, need = 10.0, 4.0
    x = m.addVar(lb=0, ub=cap, name="x")
    y = m.addVar(vtype="B", name="y")
    m.addCons(x >= need, name="demand")
    if bigm == "loose":
        m.addCons(x <= 10000.0 * y, name="link")
    elif bigm == "tight":
        m.addCons(x <= cap * y, name="link")
    else:
        m.addConsIndicator(x <= 0, binvar=y, activeone=False, name="link")
    m.setObjective(5 * y + 2 * x, "minimize")
    return m

for bigm in ("loose", "tight", "indicator"):
    mm = toy_model(bigm)
    mm.optimize()
    print(f"{bigm:10s}: obj={mm.getObjVal():.2f}  status={mm.getStatus()}")

All three variants reach the same optimal value (different representations of equivalent logical constraints). The difference lies in how tightly they are relaxed, not in the optimal value itself.

## 4. Effect (before/after) — `mk.compare_variants`

We solve the three variants of `fixed_charge` (loose / tight / indicator) with the same time limit and compare their **root dual bounds, final gaps, and node counts**. `mk.compare_variants` measures using default settings (with presolve/cuts), so we first look at those as is.

In [ ]:
df = mk.compare_variants(
    {"loose Big-M":  lambda: fc.build_model("loose"),
     "tight Big-M":  lambda: fc.build_model("tight"),
     "indicator":    lambda: fc.build_model("indicator")},
    time_limit=10)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

**Honest result**: For this scale of 8 facilities, the default setting's `root_dual`/`final_gap`/`nodes` are almost **identical** across the 3 variants. As seen in Section 1, `numerical_scale` did not fire because **presolve automatically tightens loose Big-M** (see "When It Is Ineffective" in the playbook).
The difference in the "quality of the formulation itself" only becomes visible in the **pure LP relaxation bound** (calculated in Section 1) when presolve/cuts are disabled.

In [ ]:
labels = ["loose Big-M", "tight Big-M", "indicator"]
bar_colors = [C["muted"], C["s1"], C["s2"]]
rows = [df.loc[df["variant"] == lab].iloc[0] for lab in labels]
pure_lp = [fc.lp_relaxation_bound(b) for b in ("loose", "tight", "indicator")]

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.10,
    subplot_titles=("純粋LP緩和境界(presolve/cut off) — 定式化そのものの質",
                    "既定設定でのルート双対境界(presolve/cut あり)",
                    "既定設定での探索ノード数"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, pure_lp, lambda v: f"{v:.0f}")
add_bars(2, [r["root_dual"] for r in rows], lambda v: f"{v:.0f}")
add_bars(3, [r["nodes"] for r in rows], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="Big-M排除の before / after — 定式化の質 vs 既定設定での実測", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

In [ ]:
loose, tight, ind = rows
print(f"純粋LP緩和境界(presolve/cut off): loose={pure_lp[0]:.1f} -> tight={pure_lp[1]:.1f} "
      f"(+{(pure_lp[1]/pure_lp[0]-1)*100:.0f}%) -> indicator={pure_lp[2]:.1f}")
print(f"既定設定ルート双対境界(presolve/cut あり): loose={loose['root_dual']:.1f} "
      f"-> tight={tight['root_dual']:.1f} -> indicator={ind['root_dual']:.1f}  (ほぼ同じ)")
print(f"既定設定ノード数: loose={int(loose['nodes'])} -> tight={int(tight['nodes'])} "
      f"-> indicator={int(ind['nodes'])}")

## Summary

- **The pure LP relaxation bound (presolve/cuts off) increases significantly with tight M / indicator** (1594 → 7127, +347%). Loose Big-M allowed a relaxation where "if y is slightly positive, x can produce fully," drastically lowering the lower bound. This is the difference in the "quality of the formulation itself."
- On the other hand, **the root dual bound and node count in the default settings (presolve/cuts on) were almost identical** (because presolve absorbs the difference at the scale of this repository). The fact that "the quality of the formulation" and "actual measurements with default settings" do not necessarily look the same is a practical lesson in itself.
- The difference between tight M and indicator is often small — the strength of indicator is in the fact that **"you don't need to choose a value for M"** (structurally eliminating the risk of misselection).

### Why SCIP Doesn't Do It Automatically

**It partially does.** The default presolve automatically tightens loose Big-M to some extent, so for the scale of this repository (8 facilities), the difference in final solving time and node counts is not as large as the numbers suggest (node counts for a naive B&B remain around 11 → 9 → 8, [FINDINGS §3](../../playbook/03-bigm.md)).
The difference becomes palpable when models are large-scale or have complex structures where presolve struggles. The reason SCIP cannot fully automate this is that calculating the "truly required minimum M" often requires **model semantics** (what is the maximum value of x genuinely permitted by this link?), leaving cases where general-purpose presolve rules cannot squeeze it down to a tight M. Indicator constraints structurally bypass this limitation.

### When It Is Ineffective / Cautions

- `numerical_scale` is designed to judge based on residual coefficient ratios **after** presolve. Do not hastily conclude it is "bad" just by looking at raw coefficient ratios before presolve (which would be on the order of 1e5 for a loose Big-M).
- For models with few facilities where presolve works well, the effect of applying tight M might be clear as an "improvement in LP relaxation bound," but small as an improvement in final solving time.

Related: [Methods Guide 3. Big-M Elimination](../../playbook/03-bigm.md) /
[Methods Guide 8. Condition Number and Numerical Soundness](../../playbook/08-condition.md)
(Compare the condition numbers $\kappa(A)$ for the same loose/tight Big-M in `08_condition_number.ipynb`).